In [2]:
import psutil
from pathlib import Path
import os
import shutil
import heapq

## Scan to get overall storage info
total, used, free = shutil.disk_usage("C:")

print(f"Total: {total / (1024**3):.2f} GB")
print(f"Used: {used / (1024**3):.2f} GB")
print(f"Free: {free / (1024**3):.2f} GB")

Total: 928.27 GB
Used: 852.53 GB
Free: 75.75 GB


In [ ]:
## Scan to get the largest file in the laptop#
root = Path.home() / "OneDrive"
k = 20
top_k_files = []


# for file in root.rglob("*"):
#     try:
#         if file.is_file():
#             size = file.stat().st_size
#             files.append((size, file))
#     except Exception as e:
#         print(f"Skipped {file}")
#         continue
    
try:
    for file in root.rglob("*"):
        try:
            if file.is_file():
                size = file.stat().st_size
                
                if len(top_k_files) < k:
                    heapq.heappush(top_k_files, (size, file))
                elif size > top_k_files[0][0]:
                    heapq.heapreplace(top_k_files, (size, file))
        except (PermissionError, OSError):
            pass
except Exception as e:
    print("Traversal failed:", e)
    

for size, file in sorted(top_k_files, reverse=True):
    print(f"{size / (1024**3):.2f} GB | {file}")

SyntaxError: 'continue' not properly in loop (1207687203.py, line 30)

In [ ]:
##Scan Largest files of the entire laptop
k = 20
top_k_files = []

for dirpath, dirnames, filenames in os.walk(
    Path.home(),
    topdown=True,
    onerror=lambda e: None
):
    for filename in filenames:
        path = Path(dirpath) / filename

        try:
            size = path.stat().st_size

            if len(top_k_files) < k:
                heapq.heappush(top_k_files, (size, path))
            elif size > top_k_files[0][0]:
                heapq.heapreplace(top_k_files, (size, path))

        except (PermissionError, FileNotFoundError, OSError):
            continue

for size, path in sorted(top_k_files, reverse=True):
    print(f"{size / (1024**3):.2f} GB | {path}")


44.05 GB | C:\Users\animu\AppData\Local\Docker\wsl\disk\docker_data.vhdx
5.99 GB | C:\Users\animu\AppData\Local\wsl\{13f46705-1e32-41fe-abb6-417cb37e08a5}\ext4.vhdx
4.21 GB | C:\Users\animu\Videos\Overwolf\Outplayed\Minecraft Java\Minecraft Java_01-13-2026_22-15-11-869\Minecraft Java_01-13-2026_23-45-4-916.mp4
4.11 GB | C:\Users\animu\Downloads\Office 2021 Full\Office 2021 Full\Office 2021 Full Crack.rar
4.11 GB | C:\Users\animu\Downloads\Office 2021 Full\Office 2021 Full\Office 2021 Full Crack\Office 2021 Full Crack\Cài đặt Full Crack.rar
3.11 GB | C:\Users\animu\.android\avd\Medium_Phone.avd\userdata-qemu.img.qcow2
3.03 GB | C:\Users\animu\Videos\Overwolf\Outplayed\Minecraft Java\Minecraft Java_01-13-2026_22-15-11-869\Minecraft Java_01-13-2026_22-54-1-816.mp4
3.00 GB | C:\Users\animu\.android\avd\Pixel_9_Pro_XL.avd\snapshots\default_boot\ram.img
2.89 GB | C:\Users\animu\AppData\Local\Android\Sdk\system-images\android-34\google_apis_playstore\x86_64-2\system.img
2.89 GB | C:\Users\ani

: 

In [3]:
from collections import defaultdict

ext_sizes = defaultdict(int)

for dirpath, _, filenames in os.walk(Path.home()):
    for filename in filenames:
        try:
            path = os.path.join(dirpath, filename)
            size = os.path.getsize(path)

            ext = Path(filename).suffix.lower()
            ext_sizes[ext] += size

        except OSError:
            pass

for ext, size in sorted(
    ext_sizes.items(),
    key=lambda x: x[1],
    reverse=True
)[:20]:
    print(f"{size/(1024**3):.2f} GB | {ext}")


50.18 GB | .vhdx
43.97 GB | 
31.05 GB | .dll
27.55 GB | .jar
18.11 GB | .img
16.21 GB | .zip
14.75 GB | .exe
12.76 GB | .rar
11.91 GB | .mp4
8.77 GB | .png
6.57 GB | .body
5.83 GB | .qcow2
5.80 GB | .file
5.63 GB | .mca
5.62 GB | .dat
5.51 GB | .bik
4.83 GB | .nvph
3.44 GB | .so
3.27 GB | .tif
3.09 GB | .bin


In [12]:
from collections import defaultdict
import psutil
from pathlib import Path
import os
import shutil
import heapq

class Node:
    def __init__(self, path):
        self.path = path
        self.size = 0
        self.children = []
        
    #def get_size(self): return self.size
    #def set_size(self, file_size: int): self.size = file_size

def build_size(path):
    if path.is_file():
        return path.stat().st_size
    
    total = 0
    
    for child in path.iterdir():
        total += build_size(child)
        
    return total


def build_filetree(path: Path) -> Node:
    node = Node(path)

    try:
        for child in path.iterdir():

            if child.is_file():
                try:
                    node.size += child.stat().st_size
                except (PermissionError, OSError, FileNotFoundError):
                    pass

            elif child.is_dir():
                try:
                    child_node = build_filetree(child)

                    node.children.append(child_node)

                    # Add subtree size
                    node.size += child_node.size

                except (PermissionError, OSError, FileNotFoundError):
                    pass

    except (PermissionError, OSError, FileNotFoundError):
        pass

    return node


def print_filetree(node, depth=0):
    MIN_SIZE = 1* 1024**3
    print(
        "  " * depth +
        f"{node.path.name} ({node.size/(1024**3):.2f} GB)"
    )

    for child in sorted(
        node.children,
        key=lambda x: x.size,
        reverse=True
    ):
        if (child.size >= MIN_SIZE):
            print_filetree(child, depth + 1)
        
root = Path.home().parent.parent
tree = build_filetree(root)
print_filetree(tree)

 (733.49 GB)
  Users (320.45 GB)
    animu (308.11 GB)
      AppData (188.52 GB)
        Local (154.85 GB)
          Docker (44.27 GB)
            wsl (44.19 GB)
              disk (44.05 GB)
          Packages (21.90 GB)
            PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0 (10.55 GB)
              LocalCache (10.55 GB)
                local-packages (7.17 GB)
                  Python311 (7.16 GB)
                    site-packages (7.15 GB)
                      torch (4.94 GB)
                        lib (4.83 GB)
                Local (3.33 GB)
                  pip (3.29 GB)
                    cache (3.29 GB)
                      http-v2 (3.29 GB)
                        e (2.57 GB)
                          9 (2.54 GB)
                            c (2.54 GB)
                              c (2.54 GB)
                                e (2.54 GB)
            SpotifyAB.SpotifyMusic_zpdnekdrzrea0 (7.56 GB)
              LocalCache (7.08 GB)
                Spotify (7.07 GB)
 

In [ ]:
## Scanning animu takes 8min.
## Scanning the whole Users folder takes 4min??
## Scanning the entire C drive takes 7 mins

print(root.parent.parent)

C:\
